In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path.cwd().parent / 'data' / 'interim'


In [ ]:
# Load all tables and parse dates
clients        = pd.read_csv(DATA / 'clients.csv')
transactions   = pd.read_csv(DATA / 'transactions.csv')
messages       = pd.read_csv(DATA / 'messages.csv')
templates      = pd.read_csv(DATA / 'message_templates.csv')
conversions    = pd.read_csv(DATA / 'conversions.csv')
business_units = pd.read_csv(DATA / 'business_units.csv')

clients['registration_date']  = pd.to_datetime(clients['registration_date'],  errors='coerce')
transactions['date']          = pd.to_datetime(transactions['date'],           errors='coerce')
messages['send_date']         = pd.to_datetime(messages['send_date'],          errors='coerce')
templates['date_from']        = pd.to_datetime(templates['date_from'],         errors='coerce')
templates['date_to']          = pd.to_datetime(templates['date_to'],           errors='coerce')
conversions['date']           = pd.to_datetime(conversions['date'],            errors='coerce')

tables = {
    'clients': clients, 'transactions': transactions,
    'messages': messages, 'templates': templates,
    'conversions': conversions, 'business_units': business_units,
}
for name, df in tables.items():
    print(f'{name:>18}: {len(df):,} rows, {df.shape[1]} cols')


In [ ]:
# Year range of each date column to spot outliers
date_cols = [
    (clients,      'registration_date', 'clients'),
    (transactions, 'date',              'transactions'),
    (messages,     'send_date',         'messages'),
    (templates,    'date_from',         'templates.date_from'),
    (templates,    'date_to',           'templates.date_to'),
    (conversions,  'date',              'conversions'),
]
for df, col, label in date_cols:
    years = df[col].dt.year.dropna().astype(int)
    null_rate = df[col].isna().mean()
    print(f'{label:<25}: {sorted(years.unique())} | null={null_rate:.3%}')


In [ ]:
# Primary key uniqueness checks
checks = [
    (clients,        'client_id',           'clients.client_id'),
    (clients,        'loyalty_client_id',   'clients.loyalty_client_id'),
    (transactions,   'transaction_id',      'transactions.transaction_id'),
    (messages,       'message_id',          'messages.message_id'),
    (templates,      'message_template_id', 'templates.message_template_id'),
    (conversions,    'conversion_id',       'conversions.conversion_id'),
    (business_units, 'id',                  'business_units.id'),
]
for df, pk, label in checks:
    nulls = df[pk].isna().sum()
    dups  = df[pk].duplicated().sum()
    status = 'OK' if nulls == 0 and dups == 0 else 'WARN'
    print(f'[{status}] {label:<38} nulls={nulls:,}  dups={dups:,}')


In [ ]:
# Foreign key integrity checks
fk_checks = [
    (messages['client_id'],            clients['loyalty_client_id'],     'msg.client_id -> clients.loyalty_client_id'),
    (messages['message_template_id'],   templates['message_template_id'], 'msg.template_id -> templates.template_id'),
    (conversions['message_id'],         messages['message_id'],           'conv.message_id -> messages.message_id'),
    (conversions['transaction_id'],      transactions['transaction_id'],   'conv.transaction_id -> transactions.transaction_id'),
    (conversions['client_id'],           clients['loyalty_client_id'],     'conv.client_id -> clients.loyalty_client_id'),
    (transactions['client_id'],          clients['client_id'],             'tx.client_id -> clients.client_id'),
    (transactions['business_unit_id'],   business_units['id'],             'tx.business_unit_id -> business_units.id'),
]
for fk_col, ref_col, label in fk_checks:
    missing = ~fk_col.dropna().isin(ref_col)
    n = missing.sum()
    rate = n / len(fk_col)
    print(f'{label:<55} missing={n:,} ({rate:.1%})')


In [ ]:
# Temporal consistency: events must not precede registration
tx_join = transactions.merge(
    clients[['client_id', 'registration_date']], on='client_id', how='left'
)
bad_tx = (tx_join['date'] < tx_join['registration_date']).sum()

msg_join = messages.merge(
    clients[['loyalty_client_id', 'registration_date']],
    left_on='client_id', right_on='loyalty_client_id', how='left'
)
bad_msg = (msg_join['send_date'] < msg_join['registration_date']).sum()

conv_join = conversions.merge(messages[['message_id', 'send_date']], on='message_id', how='left')
bad_conv = (conv_join['date'] < conv_join['send_date']).sum()

print(f'Transactions before registration: {bad_tx:,}  (58% register at POS during purchase)')
print(f'Messages before registration:     {bad_msg:,}')
print(f'Conversions before message sent:  {bad_conv:,}')


In [ ]:
# Full referential integrity and size summary
print('=' * 55)
print('TABLE SIZES')
print('=' * 55)
for name, df in tables.items():
    print(f'  {name:<20}: {len(df):>12,} rows')

print()
print('=' * 55)
print('MESSAGES')
print('=' * 55)
msg_clients_ok = messages['client_id'].isin(clients['loyalty_client_id'])
msg_tmpl_ok    = messages['message_template_id'].isin(templates['message_template_id'])
print(f'  known client_id:      {msg_clients_ok.mean()*100:.1f}%')
print(f'  known template_id:    {msg_tmpl_ok.mean()*100:.1f}%')
print(f'  channels: {dict(messages["channel"].value_counts())}')
print(f'  statuses: {dict(messages["status"].value_counts())}')

print()
print('=' * 55)
print('CONVERSIONS')
print('=' * 55)
conv_msg_ok = conversions['message_id'].isin(messages['message_id'])
conv_tx_ok  = conversions['transaction_id'].isin(transactions['transaction_id'])
conv_cl_ok  = conversions['client_id'].isin(clients['loyalty_client_id'])
conv_rate   = conversions['message_id'].nunique() / messages['message_id'].nunique()
print(f'  known message_id:     {conv_msg_ok.mean()*100:.1f}%')
print(f'  known transaction_id: {conv_tx_ok.mean()*100:.1f}%')
print(f'  known client_id:      {conv_cl_ok.mean()*100:.1f}%')
print(f'  conversion rate:      {conv_rate:.1%} (unique msg with conversion / total msg)')

print()
print('=' * 55)
print('TRANSACTIONS')
print('=' * 55)
tx_cl_ok = transactions['client_id'].isin(clients['client_id'])
tx_bu_ok = transactions['business_unit_id'].isin(business_units['id'])
print(f'  known client_id:      {tx_cl_ok.mean()*100:.1f}%')
print(f'  known business_unit:  {tx_bu_ok.mean()*100:.1f}%')
print(f'  NaN payed_amount:     {transactions["payed_amount"].isna().sum():,}')
